In [7]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 131 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (822 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [8]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [9]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [10]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [12]:
!ollama pull llama3.2

In [13]:
!ollama run llama3.2 "Explain LangChain in simple words"

LangChain is an open-source framework for building decentralized data workf
workflows. Here's a simplified explanation:

**Imagine a data pipeline**

Think of a data pipeline like a factory that takes raw materials (data), pr
processes them, and then sends the processed data to different destinations
destinations (e.g., storage, analysis tools).

**Decentralized data workflow**

LangChain allows you to build this pipeline using blockchain technology. He
Here's how:

1. **Data sources**: You have multiple data sources (e.g., files, databases
databases) that contain raw data.
2. **Chain links**: LangChain provides a set of "chain links" that act as t
the connectors between these data sources and other destinations (e.g., ana
analytics tools, machine learning models).
3. **Decentralized workflows**: When you create a workflow using LangChain,
LangChain, it's like creating a recipe for your data pipeline. The framewor
framework automates the process of connecting the data sources, processi

In [14]:
!pip install -q langchain langchain-community langchain-core langchain-ollama chromadb pypdf sentence-transformers

In [15]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.2"
)

response = llm.invoke(
    "What is LangChain?"
)

print(response)

LangChain is an open-source, Rust-based library that provides a set of tools and utilities for building and interacting with blockchain data storage. It's designed to simplify the process of storing, retrieving, and managing data on various blockchain platforms.

LangChain allows developers to easily interact with different blockchain protocols, such as Ethereum, Polkadot, and Solana, using a standardized interface. This makes it easier to build applications that can store, retrieve, and manage data across multiple blockchain networks.

Some key features of LangChain include:

1. Interoperability: LangChain enables seamless interaction between different blockchain platforms, allowing developers to access data from various sources in a single application.
2. Data storage: LangChain provides a flexible and scalable way to store data on blockchain networks, using protocols such as IPFS (InterPlanetary File System) or Etherscan.
3. Querying and retrieval: LangChain allows developers to que

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [17]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="""

Explain {topic}
in simple words with examples.

"""
)

In [18]:
parser = StrOutputParser()

In [19]:
chain = prompt | llm | parser

In [20]:
topics=[
    "Machine Learning",
    "LangChain",
    "Vector Database",
    "RAG",
    "Artificial Intelligence"
]


for topic in topics:

    print("\n================")
    print(topic)
    print("================")

    result = chain.invoke(
        {"topic":topic}
    )

    print(result)


Machine Learning
**What is Machine Learning?**

Machine learning (ML) is a way to teach computers to make decisions or predictions based on data, without being explicitly programmed. It's like training a child to recognize objects - you show them many examples of the object and let them figure it out for themselves!

In simple words, machine learning allows computers to:

1. Learn from data
2. Make predictions or decisions
3. Improve over time

**How does Machine Learning work?**

Imagine you have a bunch of pictures of dogs and cats. You want to teach a computer to recognize whether a new picture is a dog or a cat.

Here's how it works:

1. **Data collection**: You collect many examples of dogs and cats (images, text, audio, etc.)
2. **Training**: You show the computer these examples and let it learn from them.
3. **Model creation**: The computer creates a model that can recognize patterns in the data.
4. **Testing**: You test the model with new, unseen data to see how well it perfor

In [32]:
from langchain_core.messages import HumanMessage, AIMessage

In [33]:
chat_history=[]

In [34]:
def chat(message):

    # Add user message
    chat_history.append(
        HumanMessage(content=message)
    )

    # Prepare conversation context
    conversation = ""

    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            conversation += f"User: {msg.content}\n"
        else:
            conversation += f"AI: {msg.content}\n"


    prompt = f"""
You are a helpful AI assistant.

Conversation history:
{conversation}

User:
{message}

Answer:
"""


    response = llm.invoke(prompt)


    # Store AI response
    chat_history.append(
        AIMessage(content=response)
    )


    return response

In [37]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"]
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [38]:
!ollama pull llama3.2:1b

In [39]:
!ollama run llama3.2:1b "Say hello"

Hello. Is there something I can help you with or would you like to chat?



In [40]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.2:1b"
)

In [41]:
from langchain.tools import Tool

In [42]:
def calculator(expression):

    return str(eval(expression))


calculator_tool = Tool(
    name="Calculator",
    func=calculator,
    description="Use for mathematical calculations"
)

In [43]:
def search(query):

    return f"""
Search result for {query}

LangChain is a framework used to build LLM applications.
"""


search_tool = Tool(
    name="WebSearch",
    func=search,
    description="Search information from web"
)

In [45]:
!pip install -q langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.6/731.6 kB 25.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.16 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-core 1.5.3 which is incompatible.
langchain 0.2.16 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.10.15 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-community 0.2.16 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-core 1.5.3 which is incompatible.
langchain-community 0.2.16 requires langsmith<0.2.0,>=0.1.0, but you have langsmith 0.10.15 which is incompatible.
langchain-text-splitters 0.2.4 requires langchain-core<0.3.0,>=0

In [46]:
from langchain_core.tools import tool

In [47]:
@tool
def calculator(expression: str) -> str:
    """Useful for solving mathematical calculations."""

    try:
        return str(eval(expression))
    except Exception:
        return "Invalid calculation"

In [48]:
@tool
def web_search(query: str) -> str:
    """Useful for searching information."""

    return f"Search result for {query}: LangChain is a framework for building LLM applications."

In [50]:
from langchain_ollama import ChatOllama

chat_llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [51]:
from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """Calculate mathematical expressions."""

    try:
        return str(eval(expression))
    except:
        return "Invalid expression"


@tool
def web_search(query: str) -> str:
    """Search information from the web."""

    return f"Search result: {query} is related to LangChain and AI."

In [53]:
!pip uninstall -y langchain langchain-core langchain-community langchain-ollama langgraph

Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
Found existing installation: langchain-core 1.5.3
Uninstalling langchain-core-1.5.3:
  Successfully uninstalled langchain-core-1.5.3
Found existing installation: langchain-community 0.2.16
Uninstalling langchain-community-0.2.16:
  Successfully uninstalled langchain-community-0.2.16
Found existing installation: langchain-ollama 1.1.0
Uninstalling langchain-ollama-1.1.0:
  Successfully uninstalled langchain-ollama-1.1.0
Found existing installation: langgraph 1.2.9
Uninstalling langgraph-1.2.9:
  Successfully uninstalled langgraph-1.2.9


In [54]:
!pip install -q \
langchain==0.3.27 \
langchain-core==0.3.72 \
langchain-community==0.3.27 \
langchain-ollama==0.3.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.72 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.9 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.72 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.72 which is incompatible.


In [55]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])

time.sleep(5)

print("Ollama started")

Ollama started


In [56]:
!ollama pull llama3.2:1b

In [57]:
from langchain_ollama import ChatOllama

chat_llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [58]:
from langchain_core.tools import tool


@tool
def calculator(expression: str):
    """Calculate mathematical expressions."""

    return str(eval(expression))


@tool
def web_search(query: str):
    """Search information."""

    return f"Search result for {query}"

In [60]:
!pip uninstall -y langchain langchain-core langchain-community langchain-ollama langgraph

Found existing installation: langchain 0.3.27
Uninstalling langchain-0.3.27:
  Successfully uninstalled langchain-0.3.27
Found existing installation: langchain-core 0.3.72
Uninstalling langchain-core-0.3.72:
  Successfully uninstalled langchain-core-0.3.72
Found existing installation: langchain-community 0.3.27
Uninstalling langchain-community-0.3.27:
  Successfully uninstalled langchain-community-0.3.27
Found existing installation: langchain-ollama 0.3.6
Uninstalling langchain-ollama-0.3.6:
  Successfully uninstalled langchain-ollama-0.3.6


In [61]:
!pip install -q \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-ollama==0.1.3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.


In [1]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])

time.sleep(5)

print("Ollama running")

Ollama running


In [2]:
from langchain_ollama import ChatOllama

chat_llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [7]:
def calculator(expression):
    try:
        return str(eval(expression))
    except:
        return "Cannot calculate"


calculator_tool = Tool(
    name="Calculator",
    func=calculator,
    description="Use this tool only for math calculations like 25*4"
)

In [6]:
agent = initialize_agent(
    tools=[
        calculator_tool,
        search_tool
    ],
    llm=chat_llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

In [9]:
from langchain_ollama import ChatOllama

chat_llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [11]:
from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """Calculate mathematical expressions."""

    try:
        return str(eval(expression))
    except Exception:
        return "Invalid calculation"


@tool
def web_search(query: str) -> str:
    """Search information."""

    return f"Search result for: {query}"

In [12]:
tools = [
    calculator,
    web_search
]

llm_with_tools = chat_llm.bind_tools(tools)

In [14]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"]
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [15]:
!ollama list

NAME               ID              SIZE      MODIFIED       
llama3.2:1b        baf6a787fdff    1.3 GB    5 minutes ago     
llama3.2:latest    a80c4f17acd5    2.0 GB    57 minutes ago    


In [16]:
from langchain_ollama import ChatOllama

chat_llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [17]:
llm_with_tools = chat_llm.bind_tools(
    [
        calculator,
        web_search
    ]
)

In [18]:
response = llm_with_tools.invoke(
    "Calculate 25*4"
)

print(response)

content='' response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-08-02T14:05:36.451702371Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22915861929, 'load_duration': 7600388338, 'prompt_eval_count': 187, 'prompt_eval_duration': 11507505000, 'eval_count': 22, 'eval_duration': 3803143000, 'message': Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='calculator', arguments={'expression': '25*4'}))]), 'logprobs': None} id='run-8dc1d277-23ff-47d9-9f26-050b63142a98-0' tool_calls=[{'name': 'calculator', 'args': {'expression': '25*4'}, 'id': 'dbaaaf4f-1916-4eb2-94b3-d67acafd4eca', 'type': 'tool_call'}] usage_metadata={'input_tokens': 187, 'output_tokens': 22, 'total_tokens': 209}


In [20]:
import chromadb

In [21]:
client = chromadb.Client()
collection=client.create_collection(
    name="ai_docuements"
)

In [35]:
documents = []

for i in range(20):
    documents.append(
        f"""
        Artificial Intelligence document number {i}
        This document explains AI concepts.
        """
    )

ids = [
    str(i) for i in range(20)
]

print(documents[:2])
print(ids[:5])

['\n        Artificial Intelligence document number 0\n        This document explains AI concepts.\n        ', '\n        Artificial Intelligence document number 1\n        This document explains AI concepts.\n        ']
['0', '1', '2', '3', '4']


In [25]:
collection.add(
    documents=documents,
    ids=ids
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 105MiB/s]


In [26]:
result = collection.query(
    query_texts=[
        "Explain AI"
    ],
    n_results=3
)


result

{'ids': [['8', '11', '15']],
 'embeddings': None,
 'documents': [['\n        Artificial Intelligence document number 8\n        This document explains AI concepts.\n        ',
   '\n        Artificial Intelligence document number 11\n        This document explains AI concepts.\n        ',
   '\n        Artificial Intelligence document number 15\n        This document explains AI concepts.\n        ']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.675591230392456, 0.7038496732711792, 0.707203209400177]]}

In [27]:
collection2 = client.create_collection(
    name="metadata_docs"
)

In [28]:
for i in range(20):

    collection2.add(
        documents=[
            f"Machine learning chapter {i}"
        ],

        ids=[
            str(i)
        ],

        metadatas=[
            {
                "topic":"ML"
            }
        ]
    )

In [29]:
collection2.query(
    query_texts=[
        "learning"
    ],

    where={
        "topic":"ML"
    },

    n_results=3
)

{'ids': [['7', '13', '4']],
 'embeddings': None,
 'documents': [['Machine learning chapter 7',
   'Machine learning chapter 13',
   'Machine learning chapter 4']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'ML'}, {'topic': 'ML'}, {'topic': 'ML'}]],
 'distances': [[0.9207038879394531, 0.9279887676239014, 0.9308702349662781]]}

In [31]:
from google.colab import files

uploaded = files.upload()

Saving AI_Researcher_Paper.pdf to AI_Researcher_Paper.pdf


In [32]:
from langchain_community.document_loaders import PyPDFLoader


pdf_name=list(uploaded.keys())[0]


loader = PyPDFLoader(pdf_name)


documents = loader.load()

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)


chunks = splitter.split_documents(
    documents
)

In [34]:
from langchain_community.embeddings import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name=
    "sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_19032/2351521296.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [36]:
from langchain_community.vectorstores import Chroma


vectorstore = Chroma.from_documents(
    chunks,
    embeddings
)

In [37]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":3
    }
)

In [38]:
question="Summarize this document"

In [39]:
docs = retriever.invoke(question)

In [40]:
context="\n".join(
    [
        doc.page_content
        for doc in docs
    ]
)

In [42]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.2:1b"
)

In [43]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

print("Ollama started")

Ollama started


In [44]:
!ollama pull llama3.2:1b

In [45]:
answer = llm.invoke(final_prompt)

print(answer)

Here's a summary of the text:

ChromaDB is an open-source vector database designed for storing and retrieving embeddings (numerical representations of text) used in Retrieval-Augmented Generation (RAG) systems. These systems combine retrieval with Large Language Models to provide efficient and scalable results. ChromaDB is particularly useful for natural language processing applications that require large datasets and significant computational resources, such as speech recognition, autonomous driving, and chatbots.
